# 3장 2강: 규제 — L1(Lasso), L2(Ridge), ElasticNet

## 실습 목표

이번 실습에서는 **Ames Housing** 데이터를 이용하여 Ridge, Lasso, ElasticNet 규제가 회귀계수와 일반화 성능에 어떤 영향을 주는지 확인합니다.

- 규제가 계수의 크기를 줄이는 이유를 이해합니다.
- `alpha` 변화에 따라 Ridge의 계수 크기와 Test 성능이 어떻게 달라지는지 확인합니다.
- Lasso가 일부 계수를 정확히 0으로 만드는지 확인합니다.
- ElasticNet이 L1과 L2를 함께 사용하는 모델이라는 점을 확인합니다.
- Ridge / Lasso / ElasticNet의 계수와 Test 성능을 비교합니다.

> 이 실습에서는 3장 2강 교안에서 다룬 규제, alpha, Lasso, Ridge, ElasticNet, StandardScaler, R² 비교만 사용합니다.

## 실습 환경 / 데이터

- Python
- pandas
- numpy
- matplotlib
- scikit-learn
- 데이터: `Ames_Housing.csv`

### 분석 목표

주택 특성을 이용하여 `SalePrice`를 예측하는 회귀 문제입니다.

### 사용할 특성

규제에 따른 계수 변화를 비교하기 위해 결측치가 없는 숫자형 특성 중 다음 12개를 사용합니다.

- `OverallQual`
- `GrLivArea`
- `GarageCars`
- `GarageArea`
- `TotalBsmtSF`
- `1stFlrSF`
- `2ndFlrSF`
- `FullBath`
- `TotRmsAbvGrd`
- `YearBuilt`
- `YearRemodAdd`
- `Fireplaces`

예측 대상은 `SalePrice`입니다.

## 실습 준비

규제는 계수의 크기를 기준으로 벌점을 주기 때문에 특성의 스케일을 맞추는 과정이 중요합니다.

이번 실습에서는 다음 순서로 데이터를 준비합니다.

1. Ames Housing 데이터 불러오기
2. 사용할 특성과 `SalePrice` 선택
3. Train / Test 분리
4. `StandardScaler`를 이용한 특성 표준화

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.metrics import r2_score

house = pd.read_csv("Ames_Housing.csv")

features = [
    "OverallQual",
    "GrLivArea",
    "GarageCars",
    "GarageArea",
    "TotalBsmtSF",
    "1stFlrSF",
    "2ndFlrSF",
    "FullBath",
    "TotRmsAbvGrd",
    "YearBuilt",
    "YearRemodAdd",
    "Fireplaces"
]

X = house[features]
y = house["SalePrice"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train 크기:", X_train_scaled.shape)
print("Test 크기:", X_test_scaled.shape)

Train 크기: (1168, 12)
Test 크기: (292, 12)


In [2]:
# Ridge: 선형회귀에 L2규제를 적용하여 계수 크기를 제한하는 모델
# -> 여러 특성 중 특정 특성에 편향된 계수가 있다면 그 계수의 크기를 줄이겠다.

# L2 규제: 회귀 계수의 제곱을 모두 더한 값을 패널티로 사용하는 규제
# -> 절댓값이 큰 계수일수록 회귀계수가 굉장히 빠르게 커지기 때문에 계수를 억제하겠다

# 다중공선성: 특정 특성 사이에 선형 관계가 있어서 회귀 계수 추정이 불안정해질 수 있다.
# -> 같은 기간에 총 구매금액과 월평균 구매금액을 함께 사용한다면 비슷한 정보를 담고 있기 때문에 다중공선성 문제를 일으킬 수 있다.

# Lasso: 선형회귀에 L1규제를 적용하여 계수 크기를 제한하는 모델
# -> 여러 특성 중 특정 특성에 편향된 계수가 있다면 그 계수의 크기를 줄이겠다.

# L1 규제: 회귀계수의 절댓값을 모두 더한 값을 패널티로 사용하는 규제
# -> 절댓값이 큰 계수일수록 규제 강도를 높여서 일반적인 계수와 맞추려고 억제하겠다.

# ElasticNet: L1규제와 L2규제를 함께 적용하는 선형회귀 모델
# -> 여러 특성 중 특정 특성에 편향된 계수가 있다면 그 계수의 크기를 줄이겠다.

---

# 필수 1. Ridge와 alpha에 따른 계수 축소

## 배경

Ridge는 **L2 규제**를 사용합니다.

Ridge는 계수의 제곱합에 벌점을 주기 때문에 큰 계수를 더 강하게 억제하고, 모든 계수를 0에 가깝게 줄이는 특징이 있습니다.

## 문제 1. 여러 alpha 값에서 Ridge를 비교하세요.

### 요구사항

다음 alpha 값을 사용합니다.

```python
alpha_values = [0.001, 0.01, 0.1, 1, 10, 100]
```

각 alpha에 대해 다음을 수행합니다.

1. `Ridge(alpha=alpha)` 모델을 만듭니다.
2. Train 데이터로 학습합니다.
3. Test 데이터를 예측합니다.
4. Test R²를 계산합니다.
5. `abs(model.coef_).sum()`으로 전체 계수 크기를 확인합니다.
6. 결과를 DataFrame으로 정리합니다.

### 확인 포인트

- alpha가 커질수록 규제 강도가 강해지는가?
- alpha가 커질수록 전체 계수 크기가 작아지는가?
- alpha가 너무 커지면 Test 성능이 낮아질 수 있는가?

In [3]:
# TODO
# 여러 alpha 값에 대해 Ridge의 Test R²와 계수 크기를 비교하세요.
alpha_values = [0.001, 0.01, 0.1, 1, 10, 100]

results = []

for alpha in alpha_values:
    model = Ridge(alpha=alpha, random_state=42)

    model.fit(X_train_scaled, y_train)

    y_pred = model.predict(X_test_scaled)

    test_r2 = r2_score(y_test, y_pred)

    coef_sum = np.sum(np.abs(model.coef_))

    results.append(
        {"alpha": alpha, "Test_R2": test_r2, "sum_abs_coef": coef_sum}
    )

ridge_results_df = pd.DataFrame(results)    

print("=== Ridge alpha별 성능 및 계수 크기 비교 ===")
print(
    ridge_results_df.to_string(
        index=False,
        formatters={"Test_R2": "{:.4f}".format, "sum_abs_coef": "{:,.2f}".format},
    )
)


=== Ridge alpha별 성능 및 계수 크기 비교 ===
  alpha Test_R2 sum_abs_coef
  0.001  0.7994   102,172.59
  0.010  0.7994   102,172.46
  0.100  0.7994   102,171.10
  1.000  0.7993   102,149.83
 10.000  0.7991   101,725.43
100.000  0.7956    97,390.12


- 실행결과

| alpha | Test R2 | 계수 절대값 함 | 내용 |
|---:|---:|---:|---|
|0.001|0.7994|102172|규제가 가장 약한 설정의 값|
|0.01|0.7994|102172|앞서 규제가 가장 약했던 설정의 값과 계수가 일치|
|0.1|0.7994|102172|규제는 지속적으로 커졌지만 여전히 가장 약했던 설정의 계수와 거의 일치|
|1|0.7993|102149|계수 합이 조금 감소했으며 R2값이 낮아짐|
|10|0.7991|101725|계수 축소가 뚜렷하며, R2값이 낮아짐|
|100|0.7956|97390|이번 후보중 계수의 합이 가장 낮으며 R2값도 가장 낮음|


- 규제를 강화하면서 계수의 전체 크기가 점차 감소했으나 Test의 성능이 나아지지 않았다.

### Q1. alpha가 커질수록 Ridge 계수는 일반적으로 어떻게 변하나요?

**답변:**  
- alpha는 일반적으로 규제의 강도이며, alpha가 커질수록 Ridge 계수는 일반적으로 절대값이 작아진다.
- Ridge는 모든 계수를 한번에 0으로 바꾸지 않고 0에 가까운 방향으로 축소하려고 한다.

### Q2. alpha가 너무 크면 어떤 문제가 발생할 수 있나요?

**답변:**  
- 중요한 특성의 계수를 지나치게 줄여서 충분히 학습하지 못할 수 있다. 즉, 편향이 심해져 과소적합될 가능성이 있다.

---

# 필수 2. Lasso와 ElasticNet의 계수 변화

## 문제 1. Lasso에서 0이 되는 계수를 확인하세요.

### 배경

Lasso는 **L1 규제**를 사용하며, 중요하지 않은 특성의 계수를 정확히 0으로 만들 수 있습니다.

이번에는 규제 강도를 충분히 높여 Lasso의 변수 선택 효과를 직접 확인합니다.

### 요구사항

1. `Lasso(alpha=1000, max_iter=10000)` 모델을 만듭니다.
2. Train 데이터로 학습합니다.
3. 각 특성의 계수를 DataFrame으로 만듭니다.
4. 계수가 0인 특성의 개수를 계산합니다.
5. Test R²를 계산합니다.

### 확인 포인트

- 일부 계수가 정확히 0이 되는가?
- Ridge와 달리 Lasso가 변수 선택 효과를 갖는 이유를 이해했는가?

In [4]:
# TODO
# Lasso(alpha=1000)를 학습하고 계수와 0 계수 개수를 확인하세요.
import numpy as np
import pandas as pd
from sklearn.linear_model import Lasso
from sklearn.metrics import r2_score

# 1. Lasso(alpha=1000, max_iter=10000) 모델 생성 (수렴 안정성을 위해 random_state 설정 권장)
lasso = Lasso(alpha=1000, max_iter=10000, random_state=42)

# 2. Train 데이터로 학습
lasso.fit(X_train_scaled, y_train)

# 3. 각 특성의 계수를 DataFrame으로 생성
coef_df = pd.DataFrame({
    "특성": features,
    "계수(coef)": lasso.coef_
})

# 4. 계수가 0인 특성의 개수 계산
zero_coef_count = np.sum(lasso.coef_ == 0)
zero_features = coef_df[coef_df["계수(coef)"] == 0]["특성"].tolist()

# 5. Test R² 계산
y_pred_lasso = lasso.predict(X_test_scaled)
test_r2_lasso = r2_score(y_test, y_pred_lasso)

print("=== Lasso(alpha=1000) 특성별 회귀계수 ===")
print(coef_df.to_string(index=False, formatters={"계수(coef)": "{:,.2f}".format}))

print("\n=== 분석 요약 ===")
print(f"전체 특성 수      : {len(features)}개")
print(f"계수가 0인 특성 수: {zero_coef_count}개 ({', '.join(zero_features)})")
print(f"Test R² Score     : {test_r2_lasso:.4f}")

=== Lasso(alpha=1000) 특성별 회귀계수 ===
          특성  계수(coef)
 OverallQual 25,520.69
   GrLivArea 21,419.63
  GarageCars  6,820.21
  GarageArea  4,370.09
 TotalBsmtSF  7,305.57
    1stFlrSF  3,521.10
    2ndFlrSF      0.00
    FullBath     -0.00
TotRmsAbvGrd     83.20
   YearBuilt  6,966.92
YearRemodAdd  6,418.69
  Fireplaces  6,221.90

=== 분석 요약 ===
전체 특성 수      : 12개
계수가 0인 특성 수: 2개 (2ndFlrSF, FullBath)
Test R² Score     : 0.7960


#### 결과 해석
- OverallQual컬럼의 계수 25,520.69의 의미: 품질이 높을수록 예측 가격이 높아지는 방향과 강도
- GrLivArea컬럼의 계수 21,419.63의 의미: 전용면적의 정보도 예측 가격에 양의 계수로 반영되고 있다.
-  2ndFlrSF컬럼의 계수 0.00의 의미: 이 모델에서 2층 면적은 예측 가격에 직접적인 기여도가 0이다.
- FullBath컬럼의 계수 -0.00의 의미: 이 모델에서 욕실은 예측 가격에 직접적인 기여도가 0이다

###
- 0인 계수가 2개인 것의 의미: 전체 12개의 특성 중 2개의 특성이 0이다. 따라서 이 두 열의 값이 바뀌거나 제거되어도 나머지 10개의 열이 같다면 모델의 결과는 달라지지 않음
- 그렇다면 계수 0인 컬럼은 모델링할 때 삭제해도 무방한가?
- -> 아니오. 데이터 분할, 규제 강도의 변화, X의 변화가 있는 경우 계수가 0이 아닐수도 있으므로 함부로 삭제하면 안된다.

### Q3. Lasso에서 계수가 0이 된다는 것은 어떤 의미인가요?

**답변:**  
Lasso 모델이 해당 변수가 결과에 아무런 영향을 주지 못한다고 판단하여 예측과정에서 완전히 제외했음을 의미한다. 

## 문제 2. ElasticNet을 학습하고 L1/L2 혼합을 확인하세요.

### 배경

ElasticNet은 L1과 L2 규제를 함께 사용합니다.

`l1_ratio`는 두 규제의 비율을 조절합니다.

- `l1_ratio`가 1에 가까움 → Lasso 성격이 강함
- `l1_ratio`가 0에 가까움 → Ridge 성격이 강함

### 요구사항

1. `ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=10000)` 모델을 만듭니다.
2. Train 데이터로 학습합니다.
3. Test R²를 계산합니다.
4. 각 특성의 계수를 확인합니다.

### 확인 포인트

- `alpha`와 `l1_ratio`의 역할을 구분할 수 있는가?
- ElasticNet이 Lasso와 Ridge의 성격을 함께 가진다는 점을 설명할 수 있는가?

In [5]:
# TODO
# ElasticNet(alpha=0.1, l1_ratio=0.5)을 학습하세요.
import numpy as np
import pandas as pd
from sklearn.linear_model import ElasticNet
from sklearn.metrics import r2_score

# 1. ElasticNet 모델 생성 (alpha=0.1, l1_ratio=0.5, max_iter=10000)
elastic = ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=10000, random_state=42)

# 2. Train 데이터로 학습
elastic.fit(X_train_scaled, y_train)

# 3. Test R² 계산
y_pred_elastic = elastic.predict(X_test_scaled)
test_r2_elastic = r2_score(y_test, y_pred_elastic)

# 4. 각 특성의 계수 확인
coef_df_elastic = pd.DataFrame(
    {"특성": features, "계수(coef)": elastic.coef_}
)

print("=== ElasticNet(alpha=0.1, l1_ratio=0.5) 결과 ===")
print(f"Test R² Score: {test_r2_elastic:.4f}\n")
print("특성별 회귀계수:")
print(
    coef_df_elastic.to_string(
        index=False, formatters={"계수(coef)": "{:,.2f}".format}
    )
)

=== ElasticNet(alpha=0.1, l1_ratio=0.5) 결과 ===
Test R² Score: 0.7973

특성별 회귀계수:
          특성  계수(coef)
 OverallQual 23,514.63
   GrLivArea 12,670.77
  GarageCars  6,785.13
  GarageArea  5,137.44
 TotalBsmtSF  8,016.09
    1stFlrSF  9,237.82
    2ndFlrSF  6,873.82
    FullBath -1,895.88
TotRmsAbvGrd  2,541.48
   YearBuilt  8,072.29
YearRemodAdd  7,518.26
  Fireplaces  6,995.26


- l1_ratio = 0.5: L1항의 앞에 0.05, L2 제곱합 앞에는 0.025, r=0.5라는 의미는 두 규제의 혼합 비율을 중간으로 쓰겠다는 의미

#### 결과 해석
- OverallQual컬럼의 계수 23,514.63의 의미: 양의 계수중 가장 큰 값, 품질 정보가 가격 예측에 가장 크게 반영
- GrLivArea컬럼의 계수 12,670.77의 의미: 양의 계수 중 두번째로 큰 값, 면적도 가격예측에 크게 반영
- 2ndFlrSF컬럼의 계수 6,873.82의 의미: Lasso에서는 0이였는데 현재는 가격 예측에 기여
- FullBath컬럼의 계수 -1,895.88의 의미: 다른 입력이 같다면 가격예측에 음의 영향을 보인다.

### Q4. `alpha`와 `l1_ratio`는 각각 무엇을 조절하나요?

**답변:**  
- alpha: 전체 규제의 총 강도
- l1_ratio: 규제 안에서 L1과 L2의 비율
- -> l1_ratio가 1에 가까워질수록 Lasso, 0에 가까워질수록 Ridge 성향이 강해진다.

---

# 과제 1. 다중공선성 진단과 Ridge / Lasso / ElasticNet 비교

## 배경

같은 Train/Test 데이터에서 세 규제 모델을 비교합니다. 성취도 평가항목에 맞춰, 비교 전에 **다중공선성의 계수 해석 영향과 VIF 진단·처치 선택**을 추가합니다.

다중공선성은 입력 특성들이 서로 강한 선형 관계를 갖는 상황입니다. VIF는 한 특성을 나머지 특성으로 설명할 때 구한 결정계수 R²를 이용해 `1 / (1 - R²)`로 계산합니다. 이때 예측 대상은 `SalePrice`가 아니라 **진단할 입력 특성 하나**이며, 보조 회귀에는 절편을 포함합니다.

## 요구사항

### A. 다중공선성 진단과 처치 방향 선택

1. **Train의 입력 특성만** 이용해 특성별 VIF를 계산하고 내림차순 표 `vif_df`를 출력합니다. 기존 `X_train_scaled`를 사용해도 됩니다. 표준화는 특성 간 선형 관계 자체를 제거하지 않습니다.
2. 이번 과제에서는 `VIF < 5`: 상대적으로 낮음, `5 ≤ VIF < 10`: 주의, `VIF ≥ 10`: 높은 수준으로 분류합니다. 이는 절대적 삭제 규칙이 아닌 실습용 판단 기준입니다.
3. 주의·높은 수준인 특성을 특성명과 VIF로 제시합니다. 없으면 없다고 작성합니다. 완전한 선형 중복으로 VIF가 무한대인 경우 높은 수준으로 해석합니다. 상수 특성의 VIF는 정의하기 어려우므로 별도로 표시합니다.
4. 다중공선성이 계수의 크기·부호·안정성과 개별 변수 해석에 미치는 영향을 설명합니다.
5. 특성의 의미와 VIF 결과를 근거로 **변수 제거·결합·규제 중 한 가지 처치 방향**을 선택하고 대상·이유·한계를 설명합니다. 높은 VIF가 없으면 불필요한 제거를 하지 않는 이유와 기존 규제 비교의 목적을 설명해도 됩니다. 제거·결합의 추가 구현은 요구하지 않습니다.

### B. 기존 규제 모델 비교

다음 설정을 그대로 사용합니다.

```python
Ridge(alpha=1.0)
Lasso(alpha=1000, max_iter=10000)
ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=10000)
```

1. 세 모델을 동일한 표준화 Train 데이터로 학습합니다.
2. 동일한 Test 데이터의 R²를 계산합니다.
3. 전체 계수 크기 `abs(coef_).sum()`과 계수가 정확히 0인 개수를 계산합니다.
4. 결과를 `comparison_df`로 정리하고, 특성별 계수를 `task_coef_df`로 출력합니다.
5. Test R²와 계수 처리 차이를 함께 설명하고 Q5~Q9에 답합니다.

> Test 점수는 지정된 세 설정의 결과를 비교·해석하는 데 사용합니다. 이 점수를 보고 변수를 제거하거나 alpha를 반복 조정하지 않습니다. 실제 모델 선택·튜닝에는 Train 내부 검증 또는 CV가 필요합니다.

### 확인 포인트

- Train 입력 특성만 사용하여 VIF를 계산했는가?
- 기준과 실제 수치를 근거로 주의할 특성을 판단했는가?
- 다중공선성의 계수 해석 영향과 처치 선택 이유를 설명했는가?
- 규제가 입력 특성의 VIF 자체를 낮추는 것은 아님을 구분했는가?
- 같은 데이터에서 세 모델의 Test R²·계수 크기·0 계수 개수를 비교했는가?


In [7]:
# TODO
# A. Train 특성별 VIF 계산 및 판단
# B. 기존 세 규제 모델 비교 및 특성별 계수 표 출력
# Q5~Q9 답변 작성
import numpy as np
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.metrics import r2_score

# ==============================================================================
# A. 다중공선성 진단 (Train 특성 대상 VIF 계산)
# ==============================================================================
# X_train_scaled가 DataFrame인 경우 컬럼명을 유지하고, ndarray인 경우 DataFrame으로 변환
if isinstance(X_train_scaled, pd.DataFrame):
    X_train_vif = X_train_scaled.copy()
else:
    X_train_vif = pd.DataFrame(X_train_scaled, columns=features)

# 특성별 VIF 계산 (표준화 데이터여도 분산팽창계수 계산 가능)
vif_data = []
for i, col in enumerate(X_train_vif.columns):
    vif_val = variance_inflation_factor(X_train_vif.values, i)
    vif_data.append({"특성": col, "VIF": vif_val})

vif_df = pd.DataFrame(vif_data).sort_values(by="VIF", ascending=False).reset_index(drop=True)

# 실습용 판단 기준 적용
def classify_vif(val):
    if np.isinf(val) or val >= 10:
        return "높은 수준 (≥ 10)"
    elif val >= 5:
        return "주의 (5 ≤ VIF < 10)"
    else:
        return "상대적으로 낮음 (< 5)"

vif_df["진단"] = vif_df["VIF"].apply(classify_vif)

print("=== [A] Train 특성별 VIF 진단 결과 ===")
display(vif_df.style.format({"VIF": "{:.2f}"}))

# ==============================================================================
# B. 기존 세 규제 모델 비교
# ==============================================================================
models = {
    "Ridge": Ridge(alpha=1.0, random_state=42),
    "Lasso": Lasso(alpha=1000, max_iter=10000, random_state=42),
    "ElasticNet": ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=10000, random_state=42)
}

comparison_results = []
coef_dict = {"특성": X_train_vif.columns}

for name, model in models.items():
    # 1. 학습
    model.fit(X_train_scaled, y_train)
    
    # 2. Test R² 계산
    y_pred = model.predict(X_test_scaled)
    r2 = r2_score(y_test, y_pred)
    
    # 3. 계수 총합 크기 및 0인 계수 개수 계산
    coefs = model.coef_
    coef_abs_sum = np.abs(coefs).sum()
    zero_coef_count = int(np.sum(coefs == 0))
    
    comparison_results.append({
        "모델": name,
        "Test R²": r2,
        "전체 계수 크기 (|coef| 합)": coef_abs_sum,
        "계수가 0인 특성 수": zero_coef_count
    })
    
    coef_dict[name] = coefs

# 4. comparison_df 및 task_coef_df 출력
comparison_df = pd.DataFrame(comparison_results)
task_coef_df = pd.DataFrame(coef_dict)

print("\n=== [B-1] 세 규제 모델 비교 (comparison_df) ===")
display(comparison_df.style.format({
    "Test R²": "{:.4f}",
    "전체 계수 크기 (|coef| 합)": "{:,.2f}",
    "계수가 0인 특성 수": "{:d}"
}))

print("\n=== [B-2] 모델별 특성 회귀계수 (task_coef_df) ===")
display(task_coef_df.style.format({
    "Ridge": "{:,.2f}",
    "Lasso": "{:,.2f}",
    "ElasticNet": "{:,.2f}"
}))

=== [A] Train 특성별 VIF 진단 결과 ===


,특성,VIF,진단
0,GrLivArea,128.78,높은 수준 (≥ 10)
1,2ndFlrSF,91.62,높은 수준 (≥ 10)
2,1stFlrSF,73.59,높은 수준 (≥ 10)
3,GarageCars,5.29,주의 (5 ≤ VIF < 10)
4,GarageArea,5.03,주의 (5 ≤ VIF < 10)
5,TotalBsmtSF,3.79,상대적으로 낮음 (< 5)
6,TotRmsAbvGrd,3.33,상대적으로 낮음 (< 5)
7,OverallQual,2.69,상대적으로 낮음 (< 5)
8,YearBuilt,2.39,상대적으로 낮음 (< 5)
9,FullBath,2.23,상대적으로 낮음 (< 5)



=== [B-1] 세 규제 모델 비교 (comparison_df) ===


,모델,Test R²,전체 계수 크기 (|coef| 합),계수가 0인 특성 수
0,Ridge,0.7993,"102,149.83",0
1,Lasso,0.7960,"88,648.01",2
2,ElasticNet,0.7973,"99,258.84",0



=== [B-2] 모델별 특성 회귀계수 (task_coef_df) ===


,특성,Ridge,Lasso,ElasticNet
0,OverallQual,"25,310.89","25,520.69","23,514.63"
1,GrLivArea,"13,960.71","21,419.63","12,670.77"
2,GarageCars,"7,041.54","6,820.21","6,785.13"
3,GarageArea,"4,382.96","4,370.09","5,137.44"
4,TotalBsmtSF,"7,238.59","7,305.57","8,016.09"
5,1stFlrSF,"10,011.02","3,521.10","9,237.82"
6,2ndFlrSF,"7,098.90",0.00,"6,873.82"
7,FullBath,"-3,154.79",-0.00,"-1,895.88"
8,TotRmsAbvGrd,"1,602.13",83.20,"2,541.48"
9,YearBuilt,"8,343.12","6,966.92","8,072.29"


### Q5. Ridge와 Lasso의 가장 큰 계수 처리 차이는 무엇인가요?

**답변:**
Ridge는 0에 가깝게 규제할 수 있지만 0으로 만들지는 못한다. 반면 Lasso는 규제를 통해 계수를 0으로 만들어 feature selection 할 수 있다.

### Q6. 규제 모델의 성능을 비교할 때 Test R²뿐 아니라 계수도 함께 보는 이유는 무엇인가요?

**답변:** 세 모델의 예측 성능이 유사하더라도 계수에 따라 모델의 안정성, 복잡도 등이 매우 다르기 때문이다.

### Q7. 다중공선성은 회귀계수의 크기·부호·안정성과 개별 변수의 영향 해석에 어떤 문제를 일으킬 수 있나요?

**답변:**
비슷한 패턴을 과도하게 학습하여 계수가 과도하게 높아지거나 부호 왜곡이 발생할 수 있다. 또한 "다른 변수를 고정했을 때 해당 변수 1단위 증가에 따른 Y 변화량"이라는 개별 변수 해석이 불가능하다.

### Q8. VIF 판단 기준은 무엇이며, 계산 결과에서 주의하거나 처치를 검토할 특성은 무엇인가요? 특성명과 VIF를 근거로 답하세요.

**답변:**
변수간의 상관 관계로 판단하며, 통상적으로 VIF가 5 이상이면 주의, 10이상이면 다중공선성이 확실히 존재한다고 본다. 계산 결과 VIF 5 이상인 GarageArea, GarageCars, 1stFlrSF, 2ndFlrSF, GrLivArea의 처치를 검토해야한다.

### Q9. VIF 결과와 특성의 의미를 고려하여 변수 제거·결합·규제 중 어떤 처치 방법을 선택하겠나요? 구체적인 대상, 선택 이유와 한계를 설명하세요.

**답변:**
- 10 이상인 1stFlrSF, 2ndFlrSF, GrLivArea 중에서 1stFlrSF, 2ndFlrSF의 합이 사실상 GrLivArea(1층 면적 + 2층 면적 = 전용 면적)이므로 GrLivArea만 남기고 제거한다.
- 5 이상인 GarageCars와 GarageArea도 사실상 차고 크기라는 의미이므로 GarageArea만 남긴다.
- feature selection을 하면 주택의 구조(1층, 2층에 관한 세부 정보)가 손실될 수 있다. 

---

# 실습 마무리

아래 질문에 짧게 답해보세요.

1. 규제는 손실함수에 무엇을 추가하는 방법인가요?
- 손실함수에 계수 크기에 대한 벌점을 추가한다

2. alpha가 커지면 규제 강도는 어떻게 변하나요?
- 규제 강도가 강해진다

3. Ridge의 계수는 일반적으로 어떻게 변하나요?
- 모든 계수를 0에 가까운 방향으로 축소함, 단 0으로 만들지는 않음

4. Lasso가 변수 선택 효과를 가지는 이유는 무엇인가요?
- 일부 계수를 0으로 만들 수 있다.

5. ElasticNet의 `l1_ratio=0.5`는 어떤 의미인가요?
- L1과 L2 규제를 절반씩 섞는 설정으로 이해할 수 있다.

6. 규제 전에 특성 스케일을 맞추는 이유는 무엇인가요?
- 규제는 계수 크기를 기준으로 패널티를 부여하기 때문에 특성들의 값의 범위가 다르면 패널티가 다르게 반영될 수 있음
- 따라서 특성 스케일을 맞춘 뒤 규제를 적용한다.